In [39]:
import csv
import pandas as pd
import os
from pathlib import Path
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy.exc import IntegrityError

In [40]:
Base = declarative_base()

# Definição das tabelas com relacionamento
class Cidade(Base):
    __tablename__ = 'cidades'
    
    ID_Cidade = Column(Integer, primary_key=True)
    Cidade = Column(String(100), nullable=False)
    UF = Column(String(2), nullable=False)
    
    # Relacionamento: uma cidade tem muitos clientes
    clientes = relationship("Cliente", back_populates="cidade")
    
    def __repr__(self):
        return f"<Cidade(ID={self.ID_Cidade}, {self.Cidade}/{self.UF})>"

class Cliente(Base):
    __tablename__ = 'clientes'
    
    ID_Cliente = Column(Integer, primary_key=True)
    Nome = Column(String(50))
    Email = Column(String(50))
    Data_Nascimento = Column(String(20))
    Estado_Civil = Column(String(20))
    Genero = Column(String(20))
    Educacao = Column(String(50))
    
    # Chave estrangeira para Cidades
    ID_Cidade = Column(Integer, ForeignKey('cidades.ID_Cidade', ondelete='SET NULL'))
    
    # Relacionamento: um cliente pertence a uma cidade
    cidade = relationship("Cidade", back_populates="clientes")
    
    def __repr__(self):
        return f"<Cliente(ID={self.ID_Cliente}, Nome={self.Nome})>"

C:\Users\pcwin\AppData\Local\Temp\ipykernel_3112\4123567155.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [41]:
def criar_banco_com_relacionamento(nome_banco='../data/DBVendas.db'):
    """Cria o banco de dados com as duas tabelas e seu relacionamento"""
    try:
        engine = create_engine(f'sqlite:///{nome_banco}', echo=False)
        print("Banco de dados conectado")
    except Exception as e:
        print(f"Erro ao ler arquivo: {e}")
    
    # Dropa tabelas existentes (cuidado! Isso apaga dados)
    # Base.metadata.drop_all(engine)
    
    # Cria todas as tabelas na ordem correta
    Base.metadata.create_all(engine)
    
    print(f"✅ Banco de dados '{nome_banco}' criado com as tabelas:")
    print(f"   - cidades")
    print(f"   - clientes (com FK para cidades)")
    
    return engine

def importar_cidades_excel(arquivo_excel, engine):
    """Importa a tabela de cidades do Excel"""
    
    print("\n📊 Importando cidades do Excel...")
    
    try:
        # Lê o arquivo Excel
        df_cidades = pd.read_excel(arquivo_excel)
#        if arquivo_excel.endswith('.xlsx'):
#            df_cidades = pd.read_excel(arquivo_excel)
#        else:
#            df_cidades = pd.read_csv(arquivo_excel, delimiter='\t')
        
        print(f"   📋 Colunas encontradas: {list(df_cidades.columns)}")
        print(f"   📊 Total de cidades: {len(df_cidades)}")
        
        # Importa para o SQLite
        df_cidades.to_sql('cidades', engine, if_exists='replace', index=False)
        
        print(f"✅ Cidades importadas com sucesso!")
        
        # Verificar importação
        Session = sessionmaker(bind=engine)
        session = Session()
        total = session.query(Cidade).count()
        session.close()
        
        print(f"   💾 {total} cidades salvas no banco")
        return True
        
    except Exception as e:
        print(f"❌ Erro ao importar cidades: {str(e)}")
        return False

def importar_clientes_csv_com_validacao(arquivo_csv, engine):
    """Importa clientes validando se as cidades existem"""
    
    print("\n👥 Importando clientes do CSV...")
    
    Session = sessionmaker(bind=engine)
    session = Session()
    
    try:
        # Primeiro, criar um dicionário de IDs de cidades válidos
        cidades_validas = {cidade.ID_Cidade for cidade in session.query(Cidade.ID_Cidade).all()}
        print(f"   ✅ Cidades válidas encontradas: {len(cidades_validas)}")
        
        if not cidades_validas:
            print("   ❌ Nenhuma cidade encontrada! Importe as cidades primeiro.")
            return False
        
        # Ler CSV
        with open(arquivo_csv, 'r', encoding='utf-8') as csvfile:
            leitor_csv = csv.DictReader(csvfile)
            
            total_linhas = 0
            importados = 0
            ignorados = 0
            
            for linha in leitor_csv:
                total_linhas += 1
                
                # Verificar se a cidade existe
                id_cidade = int(linha.get('ID_Cidade', 0))
                
                if id_cidade not in cidades_validas:
                    print(f"   ⚠️  Linha {total_linhas}: ID_Cidade {id_cidade} não encontrado - registro ignorado")
                    ignorados += 1
                    continue
                
                # Criar cliente
                cliente = Cliente(
                    ID_Cliente=int(linha.get('ID_Cliente', 0)),
                    Nome=(linha.get('Primeiro Nome', '').strip() + ' ' + linha.get('Sobrenome', '').strip()),
                    Email=(linha.get('Primeiro Nome', '').strip() + '@gmail.com'),
                    Data_Nascimento=linha.get('Data Nascimento', '').strip(),
                    Estado_Civil=linha.get('Estado Civil', '').strip(),
                    Genero=linha.get('Genero', '').strip(),
                    Educacao=linha.get('Educacao', '').strip(),
                    ID_Cidade=id_cidade
                )
                
                session.add(cliente)
                importados += 1
                
                if importados % 100 == 0:
                    session.commit()
                    print(f"   📝 {importados} clientes importados...")
            
            session.commit()
        
        print(f"\n✅ Importação de clientes concluída!")
        print(f"   📊 Total processado: {total_linhas}")
        print(f"   ✅ Importados: {importados}")
        print(f"   ⚠️  Ignorados (cidade inválida): {ignorados}")
        
        return True
        
    except Exception as e:
        print(f"❌ Erro na importação: {str(e)}")
        session.rollback()
        return False
    finally:
        session.close()

def consultar_dados_com_relacionamento(engine):
    """Consulta demonstrando o relacionamento entre tabelas"""
    
    Session = sessionmaker(bind=engine)
    session = Session()
    
    print("\n" + "="*60)
    print("📊 CONSULTAS COM RELACIONAMENTO")
    print("="*60)
    
    # 1. Clientes com suas respectivas cidades (JOIN)
    print("\n1️⃣ Clientes com informações da cidade:")
    print("-" * 60)
    
    resultados = session.query(
        Cliente.ID_Cliente,
        Cliente.Nome,
        Cliente.Email,
        Cidade.Cidade,
        Cidade.UF
    ).join(Cidade, Cliente.ID_Cidade == Cidade.ID_Cidade).limit(10).all()
    
    for cliente in resultados:
        print(f"   {cliente.ID_Cliente} - {cliente.Nome} - {cliente.Email} - "
              f"Cidade: {cliente.Cidade}/{cliente.UF}")
    
    # 2. Quantidade de clientes por cidade
    print("\n2️⃣ Quantidade de clientes por cidade:")
    print("-" * 60)
    
    from sqlalchemy import func
    
    resultados = session.query(
        Cidade.Cidade,
        Cidade.UF,
        func.count(Cliente.ID_Cliente).label('total_clientes')
    ).outerjoin(Cliente).group_by(Cidade.ID_Cidade).order_by(func.count(Cliente.ID_Cliente).desc()).all()
    
    for cidade in resultados[:10]:  # Top 10
        print(f"   {cidade.Cidade}/{cidade.UF}: {cidade.total_clientes} clientes")
    
    # 3. Clientes que moram em São Paulo
    print("\n3️⃣ Clientes de São Paulo (SP):")
    print("-" * 60)
    
    clientes_sp = session.query(Cliente).join(Cidade).filter(Cidade.UF == 'SP').limit(5).all()
    for cliente in clientes_sp:
        print(f"   {cliente.Nome} - {cliente.cidade.Cidade}")
    
    session.close()

def pipeline_completo(arquivo_clientes=ARQUIVO_CLIENTES, 
                      arquivo_cidades=ARQUIVO_CIDADES,
                      banco_dados= BANCO_DADOS):
    """Pipeline completo: importa cidades primeiro, depois clientes"""
    
    print("="*60)
    print("🚀 PIPELINE COMPLETO: CIDADES → CLIENTES")
    print("="*60)
    
    # Passo 1: Criar banco de dados
    print("\n📦 Passo 1: Criando banco de dados...")
    engine = criar_banco_com_relacionamento(banco_dados)
    
    # Passo 2: Importar cidades PRIMEIRO
    print("\n📥 Passo 2: Importando tabela de CIDADES...")
    if not importar_cidades_excel(arquivo_cidades, engine):
        print("❌ Falha na importação de cidades. Pipeline interrompido.")
        return False
    
    # Passo 3: Importar clientes DEPOIS
    print("\n📥 Passo 3: Importando tabela de CLIENTES...")
    if not importar_clientes_csv_com_validacao(arquivo_clientes, engine):
        print("❌ Falha na importação de clientes.")
        return False
    
    # Passo 4: Consultas de verificação
    consultar_dados_com_relacionamento(engine)
    
    print("\n" + "="*60)
    print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
    print("="*60)
    
    return True

# Configurações
diretorio_atual = Path(os.getcwd())
ARQUIVO_CLIENTES = diretorio_atual.parent / 'planilhas' / 'Clientes.csv'      # CSV dos clientes (convertido anteriormente)
ARQUIVO_CIDADES = diretorio_atual.parent / 'planilhas' / 'Cidade.xlsx'        # Excel com as cidades
    
BANCO_DADOS = diretorio_atual.parent / 'data' / 'DBVendas.db'

print(ARQUIVO_CLIENTES)
print(ARQUIVO_CIDADES)
print(BANCO_DADOS)

# Executar pipeline
pipeline_completo(ARQUIVO_CLIENTES, ARQUIVO_CIDADES, BANCO_DADOS)

C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Clientes.csv
C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Cidade.xlsx
C:\Users\pcwin\Documents\Converte_TXT_CSV\data\DBVendas.db
🚀 PIPELINE COMPLETO: CIDADES → CLIENTES

📦 Passo 1: Criando banco de dados...
Banco de dados conectado
✅ Banco de dados 'C:\Users\pcwin\Documents\Converte_TXT_CSV\data\DBVendas.db' criado com as tabelas:
   - cidades
   - clientes (com FK para cidades)

📥 Passo 2: Importando tabela de CIDADES...

📊 Importando cidades do Excel...
   📋 Colunas encontradas: ['ID_Cidade', 'Cidade', 'UF']
   📊 Total de cidades: 31
✅ Cidades importadas com sucesso!
   💾 31 cidades salvas no banco

📥 Passo 3: Importando tabela de CLIENTES...

👥 Importando clientes do CSV...
   ✅ Cidades válidas encontradas: 31
❌ Erro na importação: (sqlite3.IntegrityError) UNIQUE constraint failed: clientes.ID_Cliente
[SQL: INSERT INTO clientes ("ID_Cliente", "Nome", "Email", "Data_Nascimento", "Estado_Civil", "Genero", "Educacao", "ID_Cidad

False

In [44]:
df = pd.read_csv('../planilhas/Clientes.csv')

duplicados = df[df.duplicated(subset=['ID_Cliente'], keep=False)]

print(duplicados)
print(f"Total duplicados: {len(duplicados)}")

Empty DataFrame
Columns: [ID_Cliente, Primeiro Nome, Sobrenome, Data Nascimento, Estado Civil, Genero, Educacao, ID_Cidade]
Index: []
Total duplicados: 0
